# Extension to All Three Iris Classes

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import time
import sys
sys.path.append("../src")

from sklearn.datasets import load_iris
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

from ansatz import build_vqc, get_prob
from training import cost, train, predict, evaluate

In [4]:
iris = load_iris()
X_full = iris.data
y_full = iris.target

scaler = MinMaxScaler((0, np.pi))
X_scaled = scaler.fit_transform(X_full)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_full, test_size=0.2, random_state=42)
print(iris.target_names)

['setosa' 'versicolor' 'virginica']


## One vs. Rest (OVR)
making three separate binary VQCs
1. Setosa (0) vs. {Versicolor (1), Virginica (2)}
2. Versicolor (1) vs. {Setosa (0), Virginica (2)}
3. Virginica (2) vs. {Setosa (0), Versicolor (1)}

argmax rule for final prediction: whichever VQC returned the highest probability

In [5]:
def make_binary_labels(y, positive_class):
    '''
    Effect:
        convert multiple labels to binary OVR
        positive class = 1, everything else = 0
    Requires:
        y: array
        positive_class: int
    '''
    return (y == positive_class).astype(int)

In [7]:
for c in range(3):
    y_binary = make_binary_labels(y_train, c)
    print(f"VQC {c} ('{iris.target_names[c]}' vs rest): positives={y_binary.sum()}, negatives={(1-y_binary).sum()}")

VQC 0 ('setosa' vs rest): positives=40, negatives=80
VQC 1 ('versicolor' vs rest): positives=41, negatives=79
VQC 2 ('virginica' vs rest): positives=39, negatives=81


## Train OVR Classifiers

In [15]:
N_LAYERS = 1
SHOTS = 500
MAXITER = 150

ovr_loss_history = {}
ovr_thetas = {}

for c in range(3):
    print(f"Training VQC {c} ('{iris.target_names[c]}' vs rest)")

    y_train_binary = make_binary_labels(y_train, c)
    t0 = time.time()
    theta_opt_c, loss_history_c = train(X_train, y_train_binary, n_layers=N_LAYERS, shots=SHOTS, maxiter=MAXITER, seed=42+c, verbose=True) # use a diff seed for each VQC
    elapsed = time.time() - t0

    ovr_loss_history[c] = loss_history_c
    ovr_thetas[c] = theta_opt_c

    # evaluate accuracy
    y_test_binary = make_binary_labels(y_test, c)
    accuracy_c, _ , _ = evaluate(X_test, y_test_binary, theta_vals=theta_opt_c, n_layers=N_LAYERS, shots=SHOTS)
    print(f"Trained VQC {c} ('{iris.target_names[c]}' vs rest): accuracy = {accuracy_c:.2%}, time elapsed = {elapsed:.2f} s")

Training VQC 0 ('setosa' vs rest)
Iter 25: loss = 0.4859
Iter 50: loss = 0.4820
Trained VQC 0 ('setosa' vs rest): accuracy = 66.67%, time elapsed = 17.69 s
Training VQC 1 ('versicolor' vs rest)
Iter 25: loss = 0.5793
Trained VQC 1 ('versicolor' vs rest): accuracy = 63.33%, time elapsed = 11.97 s
Training VQC 2 ('virginica' vs rest)
Iter 25: loss = 0.4927
Trained VQC 2 ('virginica' vs rest): accuracy = 86.67%, time elapsed = 12.56 s
